# DS 208 · Week 5 — NumPy
## Arrays, operations and indexing

Every code block from the week 5 lecture, in order, so you can run what you watched
instead of retyping it. The flood data is fetched from the course API, so there is no file
to download.

**How to use this.** Run the cells top to bottom the first time (Runtime → Run all). After
that, change things: the point of a notebook is that you can break it and find out why.

**One display difference.** NumPy 2 prints a single number as `np.float64(30.25)`; the
slides show just `30.25`. It is the same value.

A section at the end, **Beyond the lecture**, takes the same array operations further on
the 34,079 flood control budgets.


In [ ]:
import numpy as np

# Show every result in a cell, not only the last one, so each cell prints what its
# slide shows.
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

---
## Part 1 — The lecture, cell by cell

Headings carry the slide numbers shown in the deck's counter, so you can jump to whichever
one you want to replay.


### Slide 7 — A list holds any type; an array holds one numeric type


In [ ]:
import numpy as np

nums = [1, 2, 3]        # a Python list
a = np.array(nums)      # a NumPy array
a                       # array([1, 2, 3])

### Slide 8 — Why array operations are fast


The slide's diagram:

```
# a list: one pointer per number, each to its own object
[1.0, 2.0, 3.0]   → ☐→obj ☐→obj ☐→obj

# an array: one block of 8-byte floats
np.array([1.0, 2.0, 3.0])
   → [ 1.0 | 2.0 | 3.0 ]
```


Expected output:

```
8000000       # vs ~32 MB as a list
```


In [ ]:
# a million numbers, 4x less memory
big_list = [i * 0.5 for i in range(1_000_000)]
big = np.array(big_list)
big.nbytes

The list figure on the slide, checked: the list itself (its pointers) plus one Python float object per number.


In [ ]:
import sys

sys.getsizeof(big_list) + sum(sys.getsizeof(x) for x in big_list)   # about 32 MB

### Slide 9 — Measuring the memory of a list and an array


**Exercise.**

1. Predict both sizes in MB.
2. Run the cell.
3. Change 1_000_000 to 10_000. Does the ratio change?


In [ ]:
import sys

big_list = [i * 0.5 for i in range(1_000_000)]
big = np.array(big_list)

pointers = sys.getsizeof(big_list)
numbers = sum(sys.getsizeof(x) for x in big_list)

print(f"list : {(pointers + numbers) / 1e6:.1f} MB")
print(f"array: {big.nbytes / 1e6:.1f} MB")

### Slide 10 — A 60× speed-up on the same million numbers


The slide's timings, measured on the lecture laptop. Yours will differ; compare the ratio, not the exact numbers:

```
loop : 6.76 ms
numpy: 0.14 ms
```


In [ ]:
# big_list and big from the memory-layout slide
import time

def with_loop():
    total = 0.0
    for x in big_list:
        total += x
    return total

start = time.perf_counter()          # read the clock
with_loop()
loop_ms = (time.perf_counter() - start) * 1000

start = time.perf_counter()
big.sum()
np_ms = (time.perf_counter() - start) * 1000

print(f"loop : {loop_ms:.2f} ms")
print(f"numpy: {np_ms:.2f} ms")

### Slide 11 — timeit repeats a measurement and averages it


The slide's timings, measured on the lecture laptop. Yours will differ; compare the ratio, not the exact numbers:

```
loop : 6.85 ms
numpy: 0.116 ms
NumPy is 59x as fast
```


In [ ]:
# big_list, big and with_loop() from the previous slides
import timeit

def time_ms(f):
    """Average milliseconds per call of f()."""
    calls, seconds = timeit.Timer(f).autorange()
    return seconds / calls * 1000

loop_ms = time_ms(with_loop)
np_ms = time_ms(big.sum)
print(f"loop : {loop_ms:.2f} ms")
print(f"numpy: {np_ms:.3f} ms")
print(f"NumPy is {loop_ms / np_ms:.0f}x as fast")

### Slide 13 — Finding the size at which NumPy overtakes sum()


**Exercise.**

1. Predict which wins at 10 numbers.
2. Run the code (it takes a few seconds).
3. Add 1_000 to the list of sizes, to narrow down where the ratio passes 1.


In [ ]:
# time_ms(f) is the helper from the timeit slide
for n in [10, 100, 10_000, 1_000_000]:
    nums = [i * 0.5 for i in range(n)]
    arr = np.array(nums)

    def list_sum():
        return sum(nums)

    ratio = time_ms(list_sum) / time_ms(arr.sum)
    print(f"{n:>9,} numbers: NumPy is {ratio:5.1f}x as fast")

### Slide 14 — An array has a single dtype


Expected output:

```
int64
float64
<U21          # now maths is broken
```


In [ ]:
np.array([1, 2, 3]).dtype


# one float makes the whole array float
np.array([1, 2, 3.5]).dtype


# one string makes it ALL strings
np.array([1, 2, '3']).dtype

### Slide 15 — Creating arrays from ranges, zeros and lists


In [ ]:
np.zeros(3)          # [0. 0. 0.]
np.arange(0, 10, 2)  # [0 2 4 6 8]
np.linspace(0, 1, 5) # [0. .25 .5 .75 1.]
np.array([[1, 2], [3, 4]])  # 2-D

### Slide 16 — Four array-construction functions


Expected output:

```
[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]
[0. 0. 0.]   [[1. 1.]
[1. 1.]]
[1 5 4 3 3]   # seeded = repeatable
```


In [ ]:
np.arange(0, 10, 2)


np.linspace(0, 1, 5)


np.zeros(3), np.ones((2,2))



rng = np.random.default_rng(42)
rng.integers(1, 7, size=5)

### Slide 17 — reshape rearranges elements without recomputing them


Expected output:

```
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
(3, 4)        # -1 = "you figure it out"
```


In [ ]:
a = np.arange(12)
a.reshape(3, 4)


a.reshape(3, -1).shape

The next line **fails on purpose**. The slide shows:

```
ValueError: cannot reshape array of size 12 into shape (5,2)
```

It is wrapped in `try`/`except` (week 4) so the rest of the notebook keeps running. Delete the wrapper to see the full traceback.


In [ ]:
try:
    a.reshape(5, 2)
except ValueError as e:
    print("ValueError:", e)

### Slide 18 — .shape and .dtype describe size and content


In [ ]:
m = np.array([[1, 2, 3],
              [4, 5, 6]])
m.shape    # (2, 3)  -> 2 rows, 3 cols
m.dtype    # int64
m.size     # 6  total elements

### Slide 19 — Building an array and reading its shape


**Exercise.**

1. Predict the shape and the total before running.
2. Change reshape(4, 5) to reshape(2, -1). What shape results?
3. Try reshape(3, 7) and read the last line of the error.


In [ ]:
a = np.arange(1, 21)      # 1, 2, ..., 20
m = a.reshape(4, 5)

print(m)
print("shape:", m.shape)
print("size: ", m.size)
print("total:", m.sum())

### Slide 20 — Predicting the dtype NumPy chooses


**Exercise.**

1. Predict the four dtypes.
2. Predict the result of small + 50.
3. Run the code and explain any surprise.


In [ ]:
print(np.array([1, 2, 3]).dtype)
print(np.array([1, 2, 3.0]).dtype)
print(np.array([1, 2, "3"]).dtype)
print(np.array([True, 2]).dtype)

small = np.array([100, 120], dtype=np.int8)
print(small + 50)

### Slide 23 — Operators apply to every element


In [ ]:
a = np.array([1, 4, 9])

a * 2        # [ 2  8 18]
a + a        # [ 2  8 18]
np.sqrt(a)   # [1. 2. 3.]

### Slide 24 — Squaring a million numbers with a loop and with NumPy


**Exercise.**

1. Predict how many times faster NumPy is.
2. Run the code.
3. Replace x * x with x ** 0.5 and arr * arr with np.sqrt(arr). Does the gap grow or shrink?


In [ ]:
# time_ms(f) is the helper from the timeit slide
nums = [i * 0.5 for i in range(1_000_000)]
arr = np.array(nums)

def with_loop():
    return [x * x for x in nums]

def with_numpy():
    return arr * arr

print(f"loop : {time_ms(with_loop):.2f} ms")
print(f"numpy: {time_ms(with_numpy):.2f} ms")

### Slide 25 — Reducing an array to a single number


In [ ]:
temps = np.array([28, 31, 33, 29])

temps.sum()   # 121
temps.mean()  # 30.25
temps.max()   # 33
temps.std()   # 1.92

### Slide 26 — The built-in sum() is a fairer opponent


**Exercise.**

1. Rank the three from fastest to slowest before running.
2. Run the code.
3. How many times faster is arr.sum() than sum(nums)?


In [ ]:
# time_ms(f) is the helper from the timeit slide
nums = [i * 0.5 for i in range(1_000_000)]
arr = np.array(nums)

def with_loop():
    total = 0.0
    for x in nums:
        total += x
    return total

def with_builtin():
    return sum(nums)

print(f"for loop  : {time_ms(with_loop):.2f} ms")
print(f"sum(nums) : {time_ms(with_builtin):.2f} ms")
print(f"arr.sum() : {time_ms(arr.sum):.3f} ms")

### Slide 27 — Mean and standard deviation by hand and with NumPy


**Exercise.**

1. Check that both lines print the same mean and standard deviation.
2. Compare the two times.
3. Change size=100_000 to size=1_000 and run again. What happens to the ratio?


In [ ]:
# time_ms(f) is the helper from the timeit slide
rng = np.random.default_rng(1)
scores = rng.normal(75, 10, size=100_000)   # made-up exam scores
data = scores.tolist()                       # the same numbers as a list

def by_hand():
    mean = sum(data) / len(data)
    var = sum((x - mean) ** 2 for x in data) / len(data)
    return mean, var ** 0.5

def with_numpy():
    return scores.mean(), scores.std()

m1, s1 = by_hand()
m2, s2 = with_numpy()
print(f"by hand: mean {m1:.4f}, std {s1:.4f}  {time_ms(by_hand):.2f} ms")
print(f"numpy  : mean {m2:.4f}, std {s2:.4f}  {time_ms(with_numpy):.3f} ms")

### Slide 28 — axis selects the direction to collapse


In [ ]:
m = np.array([[1, 2, 3],
              [4, 5, 6]])   # 2 rows × 3 columns

### Slide 29 — Averages per student and per quiz with axis


**Exercise.**

1. Predict the shapes of per_student and per_quiz.
2. Run the code.
3. Find the student with the highest average, using .argmax().


In [ ]:
rng = np.random.default_rng(7)
grades = rng.integers(60, 101, size=(5, 4))   # 5 students x 4 quizzes
print(grades)

per_student = grades.mean(axis=1)
per_quiz = grades.mean(axis=0)
print("per student:", per_student)
print("per quiz:   ", per_quiz)
print("hardest quiz is number", per_quiz.argmin())

### Slide 30 — A scalar is stretched to fit the array


In [ ]:
prices = np.array([100, 200, 300])

prices * 1.12   # [112. 224. 336.]
prices - prices.mean()  # centred

### Slide 31 — Converting a year of temperatures at once


**Exercise.**

1. With 8,760 numbers instead of a million, predict whether the speed-up is larger or smaller.
2. Run the code.
3. Convert f back to Celsius in one line, and check that its first three values match celsius[:3].


In [ ]:
# time_ms(f) is the helper from the timeit slide
rng = np.random.default_rng(3)
celsius = rng.normal(28, 2, size=365 * 24)   # a year of hourly readings
readings = celsius.tolist()

def with_loop():
    return [c * 9 / 5 + 32 for c in readings]

def with_numpy():
    return celsius * 9 / 5 + 32

f = with_numpy()
print(f"{f.size} readings, hottest {f.max():.1f} F")
print(f"loop : {time_ms(with_loop):.3f} ms")
print(f"numpy: {time_ms(with_numpy):.3f} ms")

### Slide 35 — Slicing works in each dimension


In [ ]:
a = np.array([10, 20, 30, 40])
a[1:3]     # [20 30]

m = np.array([[1, 2], [3, 4]])
m[0, 1]    # 2  (row 0, col 1)
m[:, 0]    # [1 3]  first column

### Slide 36 — Selecting elements with a boolean mask


In [ ]:
temps = np.array([28, 31, 33, 29])

temps > 30          # [F T T F]
temps[temps > 30]  # [31 33]

### Slide 37 — Counting sixes in a million dice rolls


**Exercise.**

1. Predict roughly how many of a million rolls are sixes.
2. Run the code.
3. Change the mask to count rolls of 5 or more: (rolls >= 5).sum().


In [ ]:
# time_ms(f) is the helper from the timeit slide
rng = np.random.default_rng(0)
rolls = rng.integers(1, 7, size=1_000_000)   # a million dice rolls
roll_list = rolls.tolist()

def with_loop():
    count = 0
    for r in roll_list:
        if r == 6:
            count += 1
    return count

def with_mask():
    return (rolls == 6).sum()

print(with_loop(), with_mask())
print(f"loop: {time_ms(with_loop):.2f} ms   mask: {time_ms(with_mask):.3f} ms")

### Slide 39 — a[1:4] returns a view, not a copy


Expected output:

```
[  1 999   3   4   5]
True          # the giveaway
```


In [ ]:
a = np.array([1,2,3,4,5])
s = a[1:4]      # a VIEW
s[0] = 999

print(a)


# you never touched `a` by name
s.base is a

### Slide 40 — Fancy indexing returns a copy


Expected output:

```
[1 2 3 4 5]   # unchanged
True
```


In [ ]:
a = np.array([1,2,3,4,5])
f = a[[1,2,3]]  # a COPY
f[0] = 999

print(a)


f.base is None

### Slide 41 — A view costs almost nothing; a copy costs time


**Exercise.**

1. Predict which is faster, and by roughly how much.
2. Run the code.
3. Change 1_000_000 to 10_000_000. Which time grows?


In [ ]:
# time_ms(f) is the helper from the timeit slide
a = np.arange(1_000_000, dtype=float)

def take_view():
    return a[::2]                # every second element

def take_copy():
    return a[::2].copy()

print("the view shares a's memory:", take_view().base is a)
print(f"view: {time_ms(take_view) * 1000:8.2f} microseconds")
print(f"copy: {time_ms(take_copy) * 1000:8.2f} microseconds")

### Slide 42 — axis names the dimension to collapse


Expected output:

```
(2, 3)
21            # everything
```


In [ ]:
m = np.array([[1,2,3],
              [4,5,6]])
m.shape


m.sum()

### Slide 42 — axis names the dimension to collapse


Expected output:

```
[5 7 9]       # shape (3,)
[ 6 15]       # shape (2,)
```


In [ ]:
m.sum(axis=0)

# collapsed the 2 ROWS
# one total per column

m.sum(axis=1)

# collapsed the 3 COLUMNS
# one total per row

### Slide 43 — A smaller shape stretches when the trailing sizes are compatible


Expected output:

```
-> shape (3, 4)
```


In [ ]:
# works: trailing 4 matches 4
np.ones((3,4)) + np.array([1,2,3,4])

The next line **fails on purpose**. The slide shows:

```
ValueError: operands could not be broadcast together with shapes (3,4) (3,)
```

It is wrapped in `try`/`except` (week 4) so the rest of the notebook keeps running. Delete the wrapper to see the full traceback.


In [ ]:
# fails: trailing 4 vs 3
try:
    np.ones((3,4)) + np.array([1,2,3])
except ValueError as e:
    print("ValueError:", e)

### Slide 44 — Centring every column with broadcasting


**Exercise.**

1. Predict the three shapes printed on the first line.
2. Run the code and check the last line.
3. Try m - m.mean(axis=1) and read the error. Why do the shapes not line up?


In [ ]:
rng = np.random.default_rng(5)
m = rng.integers(0, 100, size=(4, 3))   # 4 rows, 3 columns

col_means = m.mean(axis=0)              # shape (3,)
centred = m - col_means                 # (4, 3) minus (3,)

print(m.shape, col_means.shape, centred.shape)
print(centred)
print(centred.sum(axis=0))              # each column now sums to 0

### Slide 45 — np.where as a vectorised if/else


Expected output:

```
[10  0  3  0]
```


In [ ]:
x = np.array([10, -5, 3, -8])

np.where(x < 0, 0, x)


# condition, if-true, if-false
# no loop, no append, no index

### Slide 46 — Replacing placeholder values with np.where


**Exercise.**

1. Run the code and note how many readings are missing.
2. Add print(x.mean()), the mean with the −999s left in. How wrong is it?
3. Compare the two times.


In [ ]:
# time_ms(f) is the helper from the timeit slide
rng = np.random.default_rng(9)
x = rng.normal(30, 3, size=1_000_000)       # made-up temperatures
x[rng.random(x.size) < 0.05] = -999         # 5% marked "no reading"
data = x.tolist()

def with_loop():
    return [float("nan") if v == -999 else v for v in data]

def with_where():
    return np.where(x == -999, np.nan, x)

clean = with_where()
print(f"{np.isnan(clean).sum()} missing; mean of the rest {np.nanmean(clean):.2f}")
print(f"loop: {time_ms(with_loop):.2f} ms   np.where: {time_ms(with_where):.2f} ms")

### Slide 47 — np.append in a loop is slower than a list


The slide's timings, measured on the lecture laptop. Yours will differ; compare the ratio, not the exact numbers:

```
np.append   : 14.02 ms
list, then  : 0.229 ms
np.arange   : 0.0014 ms
```


In [ ]:
# time_ms(f) is the helper from the timeit slide
def grow():                       # O(n^2)
    out = np.array([])
    for i in range(10000):
        out = np.append(out, i)
    return out

def build_list():                 # O(n)
    return np.array([i for i in range(10000)])

def direct():
    return np.arange(10000)

print(f"np.append   : {time_ms(grow):.2f} ms")
print(f"list, then  : {time_ms(build_list):.3f} ms")
print(f"np.arange   : {time_ms(direct):.4f} ms")

### Slide 50 — Summarising rainfall data


In [ ]:
rain = np.array([0, 12, 4, 33, 0, 8])

wet = rain[rain > 0]
wet.mean()          # 14.25 mm
(rain == 0).sum()   # 2 dry days

### Slide 51 — Estimating π by throwing random darts


**Exercise.**

1. Run the code and compare the two estimates.
2. Change n to 1_000, then to 3_000_000. How does the estimate change?
3. Which operations from this week does with_numpy use?


In [ ]:
# time_ms(f) is the helper from the timeit slide
rng = np.random.default_rng(42)
n = 1_000_000
x, y = rng.random(n), rng.random(n)    # n darts in a 1 x 1 square
xs, ys = x.tolist(), y.tolist()

def with_numpy():
    inside = x**2 + y**2 <= 1          # inside the quarter circle?
    return 4 * inside.mean()

def with_loop():
    hits = 0
    for i in range(n):
        if xs[i]**2 + ys[i]**2 <= 1:
            hits += 1
    return 4 * hits / n

print(with_numpy(), with_loop())
print(f"loop: {time_ms(with_loop):.0f} ms   numpy: {time_ms(with_numpy):.1f} ms")

### Slide 53 — From CSV to a NumPy array


Expected output:

```
((34079,), dtype('float64'))
1600088782392.83
```


In [ ]:
import csv, io, requests

url  = "https://portal.latarak.com/api/flood/export.csv"
resp = requests.get(url, timeout=60)
rows = list(csv.DictReader(io.StringIO(resp.text)))

budgets = np.array([float(r["budget"])
                    for r in rows])

budgets.shape, budgets.dtype


budgets.sum()

# PHP 1.6 trillion

### Slide 54 — Four summaries in four lines, without a loop


Expected output:

```
46,952,340
37,676,233
```


In [ ]:
budgets.mean()


np.median(budgets)


# mean > median => right skew

### Slide 54 — Four summaries in four lines, without a loop


Expected output:

```
1,447,499,996
3             # over PHP 1B
```


In [ ]:
budgets.max()


(budgets > 1e9).sum()


# a bool array sums as 0/1

### Slide 55 — Select rows by a condition, not a position


Expected output:

```
array([1.44750000e+09,
       1.09625523e+09,
       1.04278280e+09])
2145
```


In [ ]:
big = budgets[budgets > 1e9]
np.sort(big)[::-1]


# mid-size band: note the ()
mid = budgets[(budgets > 1e8) &
              (budgets < 5e8)]
mid.size

### Slide 56 — argsort returns positions, not values


Expected output:

```
array([18777, 33185, 28049])
Central Office
Central Office
Central Office
```


In [ ]:
order = budgets.argsort()[::-1]
order[:3]

# POSITIONS of the 3 biggest

# now look up the real records
for i in order[:3]:
    print(rows[i]["region"])

---
# Part 2 — Beyond the lecture

Everything below is new, and uses only what this week covered: masks, `np.where`,
sorting, and reductions. Three questions about where the money in this dataset goes.


## 1. The 939 zero budgets

The lecture used `np.where` to swap a placeholder value for `np.nan`. In this dataset,
939 projects list a budget of exactly 0. Mark them as missing the same way, and watch what
happens to `.mean()`.


In [ ]:
b = np.where(budgets == 0, np.nan, budgets)

print("zeros marked missing:", np.isnan(b).sum())
print("b.mean()            :", b.mean())

`nan`. One missing value makes the whole sum unknown, so the mean is unknown too. NumPy
has `nan`-aware versions of the reductions that skip the gaps:


In [ ]:
print(f"mean,  zeros counted : {budgets.mean():>14,.0f}")
print(f"mean,  zeros skipped : {np.nanmean(b):>14,.0f}")
print(f"median, zeros counted: {np.median(budgets):>14,.0f}")
print(f"median, zeros skipped: {np.nanmedian(b):>14,.0f}")

Counting the zeros as real budgets pulls the mean down from about PHP 48.3 million to
PHP 47.0 million, and the median from 39.2 million to 37.7 million.

Which of the two is right depends on what a zero *means*. A mask answers that: compare
each project's status against the zero mask.


In [ ]:
status = np.array([r["status"] for r in rows])

np.unique(status[budgets == 0], return_counts=True)

Every zero belongs to a project that is **For Procurement** (876) or **Terminated**
(63). No contract has been awarded, or the contract was cancelled, so there is no budget to
report. The zeros are not cheap projects; they are projects without a price yet. Counting
them as PHP 0 understates the typical project.


## 2. How concentrated is the spending?

The lecture found that the top 1% of projects hold 6.5% of all spending. The same three
steps (sort biggest first, keep the first *k*, divide by the total) work for any share.


In [ ]:
biggest_first = np.sort(budgets)[::-1]
total = budgets.sum()

for pct in [1, 10, 50]:
    k = round(len(budgets) * pct / 100)
    share = biggest_first[:k].sum() / total
    print(f"top {pct:>2}% ({k:>6,} projects) hold {share:6.1%} of the money")

Roughly: the top 1% hold 6.5%, the top 10% hold 30%, and the larger half of the projects
hold about 84% of the money. The smaller 17,000 projects share the remaining 16%.


## 3. Spending per year, with one mask per year

Each project has a `year`. A mask per year selects that year's budgets; `.sum()` totals
them. `np.unique` lists the years so nothing is typed by hand.


In [ ]:
years = np.array([int(r["year"]) for r in rows])

for y in np.unique(years):
    in_year = years == y
    print(f"{y}  {in_year.sum():>5,} projects   PHP {budgets[in_year].sum() / 1e9:6.1f} billion")

Spending climbs to a peak of about PHP 369 billion across 5,553 projects in 2024.
Read the last rows with care: 2025 is a partial year in this export (its latest start
date is in September 2025), and 2026 has a single project. A falling line at the end of a chart is often an incomplete year, not a decline.

---

## What to take from this

1. **A zero is not always a number.** Here it means "no contract yet", and treating it as
   PHP 0 shifts the mean and the median.
2. **`nan` spreads.** One `nan` turns `.mean()` into `nan`; the `np.nan*` functions skip the
   gaps on purpose.
3. **A mask is a question.** `status[budgets == 0]` asked *which* projects have zero
   budgets, and the answer changed what the zeros mean.
4. **Sorting plus slicing answers "how concentrated".** Three lines give the share of the
   top 1%, 10% or 50%.

---

*Data: DPWH flood control projects 2016–2026, mirrored by BetterGov.ph from the DPWH
transparency portal, CC0-1.0 (public domain). Served by the course API at*
`/api/flood` *— see* `/api/flood` *itself for the full endpoint list.*
